# Perception Run

In [1]:
"""Everything that runs today, in the shape a notebook would use it.

No affect model, no encoder, no embodiment, no recorder — so nothing folds, expresses or
writes. What does work is the whole producer side: utterances in, affect evidence out, in
either representation, with ground truth attached and scored. That is §12.5's benchmark.
"""

import asyncio

from asa.core.affect import AffectVector, Utterance
from asa.core.representations import BASIC4, EKMAN6
from asa.perception.decode_keyword import (
    BASIC4_KEYWORDS,
    EKMAN6_KEYWORDS,
    KeywordDecoder,
)

# A labelled set: the sentence, and the axis it was written to convey. `intended` is a real
# field on Utterance and needs no adapter — RandomEmotion (step 13) will fill it the same way.
LABELLED = [
    ("I am so happy", "happiness"),
    ("I was gutted", "sadness"),
    ("that is absolutely revolting", "anger_disgust"),
    ("I was terrified", "fear_surprise"),
    ("I'm furious about it", "anger_disgust"),
    ("I just got the job!", "happiness"),        # states a fact, not an affect
    ("what time is the meeting", None),          # genuinely neutral
]


def dominant(vector: AffectVector, threshold: float = 0.15) -> str | None:
    """The five-way quantiser §12.5 describes: the strongest axis, or neutral."""
    top = max(vector.values, key=lambda axis: vector.values[axis])
    return top if vector.values[top] > threshold else None


async def score(decoder: KeywordDecoder, label: str) -> None:
    hits = 0
    print(f"\n=== {label} ===")
    for text, intended_axis in LABELLED:
        intended = (AffectVector(decoder._representation.id, {intended_axis: 1.0})
                    if intended_axis else None)
        utterance = Utterance(text=text, source="input:benchmark", intended=intended)

        observation = await decoder.decode(utterance)
        got = dominant(observation.affect)
        # only score rows this representation can express an intention for
        scorable = intended_axis is None or intended_axis in decoder._representation.axes
        hit = scorable and got == intended_axis
        hits += bool(hit)
        mark = "ok " if hit else ("-- " if scorable else "n/a")
        print(f"  {mark} {text!r:34} intended={intended_axis!s:14} decoded={got}")
    print(f"  dominant-axis agreement: {hits}/{len(LABELLED)}")


async def main() -> None:
    await score(KeywordDecoder(BASIC4, BASIC4_KEYWORDS), "basic4/1")
    await score(KeywordDecoder(EKMAN6, EKMAN6_KEYWORDS), "ekman6/1")


asyncio.run(main())


RuntimeError: asyncio.run() cannot be called from a running event loop